# VQA Model Error Analysis
Interactive notebook for analyzing model predictions and failure cases

In [ ]:
# Setup
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import numpy as np

# If running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/vqa-med
except:
    pass

sns.set_style('whitegrid')

## 1. Load Evaluation Results

In [ ]:
# Set paths
EVAL_DIR = Path('outputs/evaluation')  # or Path('/content/drive/MyDrive/vqa-med-evaluation')
IMAGE_DIR = Path('data/raw/VQA-RAD/images')

# Load results
results_df = pd.read_csv(EVAL_DIR / 'detailed_results.csv')
failures_df = pd.read_csv(EVAL_DIR / 'failure_cases.csv')

with open(EVAL_DIR / 'metrics.json', 'r') as f:
    metrics = json.load(f)

print(f"Total samples: {len(results_df)}")
print(f"Correct predictions: {results_df['correct'].sum()}")
print(f"Incorrect predictions: {(~results_df['correct']).sum()}")
print(f"Overall accuracy: {metrics['overall_accuracy']:.2f}%")

## 2. Overall Performance

In [ ]:
# Display metrics
print("="*60)
print("Performance Summary")
print("="*60)
print(f"Overall Accuracy: {metrics['overall_accuracy']:.2f}%")
print(f"Average Confidence: {metrics['avg_confidence']:.2%}")
print(f"Correct Predictions Confidence: {metrics['correct_avg_confidence']:.2%}")
print(f"Incorrect Predictions Confidence: {metrics['incorrect_avg_confidence']:.2%}")

print("\nAccuracy by Question Type:")
for qt, acc in sorted(metrics['question_type_accuracy'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {qt:20s}: {acc:6.2f}%")

## 3. Error Analysis by Question Type

In [ ]:
# Errors by question type
errors_by_qt = results_df[~results_df['correct']].groupby('question_type').size().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
errors_by_qt.plot(kind='bar')
plt.title('Number of Errors by Question Type')
plt.xlabel('Question Type')
plt.ylabel('Number of Errors')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Show error rate per question type
qt_stats = results_df.groupby('question_type').agg({
    'correct': ['sum', 'count']
})
qt_stats.columns = ['correct', 'total']
qt_stats['error_rate'] = 100 * (1 - qt_stats['correct'] / qt_stats['total'])
qt_stats = qt_stats.sort_values('error_rate', ascending=False)

print("\nError Rate by Question Type:")
print(qt_stats)

## 4. Common Prediction Errors

In [ ]:
# Most common incorrect predictions
wrong_predictions = results_df[~results_df['correct']]

print("Most Common Wrong Predictions:")
print("\nGround Truth → Predicted (Count)")
error_pairs = wrong_predictions.groupby(['ground_truth', 'predicted']).size().sort_values(ascending=False).head(20)
for (gt, pred), count in error_pairs.items():
    print(f"  '{gt}' → '{pred}': {count}")

## 5. Confidence Analysis

In [ ]:
# Confidence distribution for correct vs incorrect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
correct_conf = results_df[results_df['correct']]['confidence']
incorrect_conf = results_df[~results_df['correct']]['confidence']

axes[0].hist(correct_conf, bins=50, alpha=0.6, label='Correct', color='green')
axes[0].hist(incorrect_conf, bins=50, alpha=0.6, label='Incorrect', color='red')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence Distribution')
axes[0].legend()

# Box plot
results_df.boxplot(column='confidence', by='correct', ax=axes[1])
axes[1].set_xlabel('Correct Prediction')
axes[1].set_ylabel('Confidence')
axes[1].set_title('Confidence by Correctness')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"Mean confidence (correct): {correct_conf.mean():.3f}")
print(f"Mean confidence (incorrect): {incorrect_conf.mean():.3f}")
print(f"Median confidence (correct): {correct_conf.median():.3f}")
print(f"Median confidence (incorrect): {incorrect_conf.median():.3f}")

## 6. High-Confidence Errors (Most Concerning)

In [ ]:
# High confidence but wrong predictions (model is very confident but wrong)
high_conf_errors = failures_df.head(20)

print("Top 20 High-Confidence Errors:")
print("="*80)
for idx, row in high_conf_errors.iterrows():
    print(f"\n{idx+1}. [{row['question_type']}] Confidence: {row['confidence']:.2%}")
    print(f"   Q: {row['question']}")
    print(f"   Predicted: {row['predicted']} | Ground Truth: {row['ground_truth']}")
    print(f"   Image: {row['image']}")

## 7. Visualize Failure Cases with Images

In [ ]:
def show_failure_case(idx, failures_df, image_dir):
    """Display a failure case with image."""
    row = failures_df.iloc[idx]
    
    # Load image
    img_path = image_dir / row['image']
    if img_path.exists():
        img = Image.open(img_path)
        
        plt.figure(figsize=(10, 8))
        plt.imshow(img, cmap='gray')
        plt.axis('off')
        plt.title(f"Question Type: {row['question_type']} | Confidence: {row['confidence']:.2%}", 
                  fontsize=12, pad=20)
        
        # Add text below image
        info_text = f"Q: {row['question']}\n\n"
        info_text += f"Predicted: {row['predicted']}\n"
        info_text += f"Ground Truth: {row['ground_truth']}"
        
        plt.text(0.5, -0.1, info_text, transform=plt.gca().transAxes,
                fontsize=11, verticalalignment='top', ha='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"Image not found: {img_path}")

# Show first 5 high-confidence failures
print("Visualizing High-Confidence Errors:\n")
for i in range(min(5, len(failures_df))):
    show_failure_case(i, failures_df, IMAGE_DIR)

## 8. Analyze Specific Question Type

In [ ]:
# Choose a question type to analyze
QUESTION_TYPE = 'ABN'  # Change this to analyze different types

qt_results = results_df[results_df['question_type'] == QUESTION_TYPE]
qt_errors = qt_results[~qt_results['correct']]

print(f"Analysis for Question Type: {QUESTION_TYPE}")
print("="*60)
print(f"Total samples: {len(qt_results)}")
print(f"Correct: {qt_results['correct'].sum()}")
print(f"Incorrect: {(~qt_results['correct']).sum()}")
print(f"Accuracy: {100 * qt_results['correct'].mean():.2f}%")

print(f"\nMost Common Errors:")
error_pairs = qt_errors.groupby(['ground_truth', 'predicted']).size().sort_values(ascending=False).head(10)
for (gt, pred), count in error_pairs.items():
    print(f"  '{gt}' → '{pred}': {count}")

# Show answer distribution for this question type
print(f"\nAnswer Distribution:")
print(qt_results['ground_truth'].value_counts().head(10))

## 9. Interactive Exploration

In [ ]:
# Filter and explore specific cases
def explore_cases(question_type=None, correct=None, min_confidence=None, max_confidence=None):
    """Filter results based on criteria."""
    filtered = results_df.copy()
    
    if question_type:
        filtered = filtered[filtered['question_type'] == question_type]
    
    if correct is not None:
        filtered = filtered[filtered['correct'] == correct]
    
    if min_confidence:
        filtered = filtered[filtered['confidence'] >= min_confidence]
    
    if max_confidence:
        filtered = filtered[filtered['confidence'] <= max_confidence]
    
    print(f"Found {len(filtered)} cases")
    return filtered

# Example: Find all incorrect predictions with high confidence (>80%) for MODALITY questions
cases = explore_cases(question_type='MODALITY', correct=False, min_confidence=0.8)
print("\nSample cases:")
print(cases[['question', 'predicted', 'ground_truth', 'confidence']].head(10))

## 10. Save Insights

In [ ]:
# Save your findings
insights = {
    'overall_accuracy': metrics['overall_accuracy'],
    'weakest_question_types': [],
    'most_common_errors': [],
    'key_observations': []
}

# Add your observations here
insights['key_observations'] = [
    "Model struggles with open-ended questions",
    "High confidence on yes/no questions",
    "Confusion between similar anatomical terms",
    # Add more as you discover
]

with open(EVAL_DIR / 'analysis_insights.json', 'w') as f:
    json.dump(insights, f, indent=2)

print("Insights saved!")